In [1]:
import pandas as pd
import duckdb

df_sensor_status = pd.DataFrame({
    "device_id": [
        "R34", "R34", "R34", "R34", "R34",
        "R34", "R34", "R34", "R34", "R34",
        "R36", "R36", "R36", "R36", "R36",
        "R36", "R36"
    ],
    "log_time": [
        "2026-07-25 10:00:00",
        "2026-07-25 10:01:00",
        "2026-07-25 10:02:00",
        "2026-07-25 10:03:00",
        "2026-07-25 10:04:00",
        "2026-07-25 10:05:00",
        "2026-07-25 10:07:00",
        "2026-07-25 10:08:00",
        "2026-07-25 10:09:00",
        "2026-07-25 10:10:00",

        "2026-07-25 10:00:00",
        "2026-07-25 10:01:00",
        "2026-07-25 10:02:00",
        "2026-07-25 10:03:00",
        "2026-07-25 10:04:00",
        "2026-07-25 10:05:00",
        "2026-07-25 10:06:00"
    ],
    "sensor_status": [
        "NORMAL",
        "MISSING",
        "MISSING",
        "MISSING",
        "NORMAL",
        "MISSING",
        "MISSING",
        "MISSING",
        "MISSING",
        "NORMAL",

        "MISSING",
        "MISSING",
        "NORMAL",
        "MISSING",
        "MISSING",
        "MISSING",
        "MISSING"
    ]
})

df_sensor_status["log_time"] = pd.to_datetime(
    df_sensor_status["log_time"]
)

df_sensor_status

,device_id,log_time,sensor_status
0,R34,2026-07-25 10:00:00,NORMAL
1,R34,2026-07-25 10:01:00,MISSING
2,R34,2026-07-25 10:02:00,MISSING
3,R34,2026-07-25 10:03:00,MISSING
4,R34,2026-07-25 10:04:00,NORMAL
5,R34,2026-07-25 10:05:00,MISSING
6,R34,2026-07-25 10:07:00,MISSING
7,R34,2026-07-25 10:08:00,MISSING
8,R34,2026-07-25 10:09:00,MISSING
9,R34,2026-07-25 10:10:00,NORMAL


# SQL Daily Review：连续缺失区间

## 题目背景

设备每分钟记录一次传感器状态：

- `NORMAL`：数据正常；
- `MISSING`：数据缺失。

现在需要识别每台设备中，持续时间较长的连续缺失区间。

## 题目要求

找出每台设备中，**连续至少 3 条记录为 `MISSING` 的区间**。

### 连续判断规则

两条缺失记录属于同一个区间，必须满足：

```text
当前缺失时间 = 上一条缺失时间 + 1 分钟
```

例如：

```text
10:05  MISSING
10:07  MISSING
```

虽然这两条记录在筛选后的缺失数据中相邻，但中间缺少 `10:06`，因此不能认为是连续缺失。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `missing_start_time` | 连续缺失开始时间 |
| `missing_end_time` | 连续缺失结束时间 |
| `missing_records` | 连续缺失记录数 |

### 最终排序

按照以下顺序排列：

1. `device_id` 升序；
2. `missing_start_time` 升序。

## 解题要求

- 先筛选 `sensor_status = 'MISSING'`；
- 使用 `LAG()` 获取上一条缺失记录的时间；
- 标记每个连续缺失区间的起点；
- 使用累计求和生成区间编号；
- 按设备和区间编号聚合；
- 只保留 `missing_records >= 3` 的区间；
- 使用 CTE 分步骤完成。

In [11]:
query = """
WITH previous_table AS (
    SELECT
        device_id,
        log_time,
        sensor_status,
        LAG(log_time) OVER (
            PARTITION BY device_id
            ORDER BY log_time
        ) AS previous_time
    FROM df_sensor_status
    WHERE sensor_status = 'MISSING'
),

missing_start AS (
    SELECT
        device_id,
        log_time,
        sensor_status,
        previous_time,
        CASE
            WHEN previous_time IS NULL
                 OR log_time <> previous_time + INTERVAL 1 MINUTE
            THEN 1
            ELSE 0
        END AS missing_start_sign
    FROM previous_table
),

phase_sign_table AS (
    SELECT
        device_id,
        log_time,
        sensor_status,
        previous_time,
        missing_start_sign,
        SUM(missing_start_sign) OVER (
            PARTITION BY device_id
            ORDER BY log_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )::INTEGER AS phase_sign
    FROM missing_start
),

count_table AS (
    SELECT
        device_id,
        MIN(log_time) AS missing_start_time,
        MAX(log_time) AS missing_end_time,
        COUNT(*) AS missing_records
    FROM phase_sign_table
    GROUP BY
        device_id,
        phase_sign
)

SELECT *
FROM count_table
WHERE missing_records >= 3
ORDER BY
    device_id,
    missing_start_time;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,missing_start_time,missing_end_time,missing_records
0,R34,2026-07-25 10:01:00,2026-07-25 10:03:00,3
1,R34,2026-07-25 10:07:00,2026-07-25 10:09:00,3
2,R36,2026-07-25 10:03:00,2026-07-25 10:06:00,4
